# Experiment B.04 — full serial run
Launch ID first. Launch OOD only after ID is complete and the selected physical GPU is idle.

In [ ]:
import os,subprocess
from pathlib import Path
GPU="4"
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; N=Path.home()/"stage1-native"; EA=Path.home()/"experiment_a"; GATE=EA/"analysis/experiment_a_to_b_gate.json"; OUT=Path.home()/"experiment_b"; MAN=OUT/"experiment_b_manifest.csv"; SCENE="id"; PY=Path.home()/"venv-stage1-id/bin/python"; log=OUT/"experiment_b_id.log"; pidfile=OUT/"experiment_b_id.pid"
gpu_state=subprocess.run(["nvidia-smi","-i",GPU,"--query-gpu=memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print("Physical GPU",GPU,"state:",gpu_state)
used,util=[int(value.strip()) for value in gpu_state.split(',')]; assert used<500 and util<5,f"STOP: physical GPU {GPU} is not idle: {gpu_state}"
env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_experiment_b","--config",str(R/"async_vla_benchmark/configs/experiment_b.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene",SCENE,"--dispatch-gate",str(GATE),"--resume","--verbose"]
fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)+"\n"); print("launched Experiment B ID on physical GPU",GPU,proc.pid,log)

In [ ]:
import csv
def alive(pid):
 state=subprocess.run(["ps","-p",str(pid),"-o","stat="],capture_output=True,text=True).stdout.strip(); return bool(state) and not state.startswith("Z")
rows=list(csv.DictReader(open(OUT/"experiment_b_episode_results.csv"))) if (OUT/"experiment_b_episode_results.csv").exists() else []; ids={row["run_id"] for row in rows}; print(f"episodes {len(ids)}/64; remaining {64-len(ids)}")
for scene in ("id","ood"):
 path=OUT/f"experiment_b_{scene}.pid"; pid=int(path.read_text()) if path.exists() else -1; print(scene,"alive=",alive(pid),"pid=",pid); log=OUT/f"experiment_b_{scene}.log"; print("\n".join(log.read_text(errors="replace").splitlines()[-8:]) if log.exists() else "not started")

In [ ]:
rows=list(csv.DictReader(open(OUT/"experiment_b_episode_results.csv"))) if (OUT/"experiment_b_episode_results.csv").exists() else []; id_rows=[row for row in rows if row["scene_condition"]=="id" and row.get("status","").startswith("ok")]; assert len({row["run_id"] for row in id_rows})==16,"STOP: ID must be 16/16 valid before OOD"
gpu_state=subprocess.run(["nvidia-smi","-i",GPU,"--query-gpu=memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); used,util=[int(value.strip()) for value in gpu_state.split(',')]; assert used<500 and util<5,f"STOP: physical GPU {GPU} is not idle: {gpu_state}"
SCENE="ood"; PY=Path.home()/"venv-stage1-ood/bin/python"; log=OUT/"experiment_b_ood.log"; pidfile=OUT/"experiment_b_ood.pid"; env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONPATH":str(P),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+env.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+env.get("LD_LIBRARY_PATH","")})
cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_experiment_b","--config",str(R/"async_vla_benchmark/configs/experiment_b.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene",SCENE,"--dispatch-gate",str(GATE),"--resume","--verbose"]
fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)+"\n"); print("launched Experiment B OOD on physical GPU",GPU,proc.pid,log)

In [ ]:
rows=list(csv.DictReader(open(OUT/"experiment_b_episode_results.csv"))) if (OUT/"experiment_b_episode_results.csv").exists() else []; ids={row["run_id"] for row in rows}; print(f"episodes {len(ids)}/64; remaining {64-len(ids)}")
for scene in ("id","ood"):
 path=OUT/f"experiment_b_{scene}.pid"; pid=int(path.read_text()) if path.exists() else -1; print(scene,"alive=",alive(pid),"pid=",pid); log=OUT/f"experiment_b_{scene}.log"; print("\n".join(log.read_text(errors="replace").splitlines()[-8:]) if log.exists() else "not started")
print("Checkpoint ~/experiment_b off-machine regularly. Rerun the matching launch cell after interruption; --resume skips valid episodes.")